# Production Governance

### Project Overview & Objective
This notebook demonstrates how the fraud engine works in real operations when a customer makes a payment. A live scoring function is created, 3 action buckets are established (approve, verify, or review), the system architecture is mapped, and operational rules for monitoring and updating the model over time are defined.


### 1. Load Saved Production Model and Pipeline Tools
Loads the saved champion Random Forest model, numerical scaler, and feature list from model/.


In [3]:
import os
import json
import pandas as pd
import numpy as np
import joblib

model_dir = os.path.join('..', 'model')
rf_model = joblib.load(os.path.join(model_dir, 'random_forest_champion.joblib'))
scaler = joblib.load(os.path.join(model_dir, 'scaler.joblib'))
feature_cols = joblib.load(os.path.join(model_dir, 'feature_names.joblib'))

with open(os.path.join(model_dir, 'evaluation_summary.json'), 'r') as f:
    eval_summary = json.load(f)

print('Production inference engine initialized.')
print(f'Total input features expected: {len(feature_cols)}')

Production inference engine initialized.
Total input features expected: 33


### 2. Create Instant Payment Risk Scoring Function
When a customer taps 'Pay', the system has less than a tenth of a second to approve or hold the transaction. This function takes a single payment request, calculates the risk score instantly, and lists plain-English reasons why the payment looks suspicious.


In [4]:
def score_transaction_payload(payload):
    """
    Scores a live transaction payload and surfaces top contributing risk signals.
    """
    amt = float(payload.get('amount', 0.0))
    abs_amt = abs(amt)
    base_avg = float(payload.get('avg_transaction_amount_30d', 100.0))
    base_avg = max(base_avg, 1.0)
    
    ratio = abs_amt / base_avg
    vel_1h = int(payload.get('transaction_velocity_1h', 0))
    is_new_dev = int(payload.get('is_new_device', 0))
    is_cb = int(payload.get('is_cross_border', 0))
    is_neg = 1 if amt < 0 else 0
    acc_age = int(payload.get('account_age_days', 180))
    hour = int(payload.get('hour_of_day', 12))
    
    # Construct feature row
    row = {col: 0 for col in feature_cols}
    row['log_amount'] = np.log1p(abs_amt)
    row['log_amount_to_avg_ratio'] = np.log1p(ratio)
    row['log_avg_30d'] = np.log1p(base_avg)
    row['transaction_velocity_1h'] = vel_1h
    row['is_new_device'] = is_new_dev
    row['is_cross_border'] = is_cb
    row['is_negative_amount'] = is_neg
    row['account_age_days'] = acc_age
    row['hour_of_day'] = hour
    
    # Set dummy flags
    for prefix, val in [
        ('account_type_', payload.get('account_type', 'Individual')),
        ('kyc_tier_', payload.get('kyc_tier', 'Verified')),
        ('channel_', payload.get('channel', 'Mobile App')),
        ('merchant_category_', payload.get('merchant_category', 'Retail'))
    ]:
        col_name = f"{prefix}{val}"
        if col_name in row:
            row[col_name] = 1
            
    input_df = pd.DataFrame([row])[feature_cols]
    risk_prob = float(rf_model.predict_proba(input_df)[0, 1])
    
    # Identify contributing signals
    signals = []
    if ratio >= 5.0:
        signals.append(f'Excessive spend deviation ({ratio:.1f}x personal 30-day baseline)')
    if vel_1h >= 2:
        signals.append(f'High transaction velocity ({vel_1h} transactions in trailing 1h)')
    if is_new_dev == 1:
        signals.append('First-time access from unverified hardware device ID')
    if is_cb == 1:
        signals.append('Cross-border transaction originating outside home jurisdiction')
    if payload.get('merchant_category') in ['Wire Transfer', 'Payroll Transfer', 'Crypto Exchange']:
        signals.append(f'High-risk merchant category ({payload.get("merchant_category")})')
    if not signals:
        signals.append('Nominal transaction parameters aligned with customer profile')
        
    return {
        'transaction_id': payload.get('transaction_id', 'TXN_TEST_001'),
        'fraud_risk_score': round(risk_prob, 4),
        'risk_level': 'High' if risk_prob >= 0.50 else ('Medium' if risk_prob >= 0.15 else 'Low'),
        'contributing_signals': signals
    }

# Test with a simulated suspicious payment payload
test_sample = {
    'transaction_id': 'TXN_SIM_9874',
    'amount': 450000.0,
    'avg_transaction_amount_30d': 15000.0,
    'account_type': 'Individual',
    'kyc_tier': 'Basic',
    'channel': 'API/Integration',
    'merchant_category': 'Wire Transfer',
    'is_new_device': 1,
    'is_cross_border': 1,
    'transaction_velocity_1h': 3
}

res = score_transaction_payload(test_sample)
print('Live Scoring Test Output')
print(json.dumps(res, indent=2))

Live Scoring Test Output
{
  "transaction_id": "TXN_SIM_9874",
  "fraud_risk_score": 0.594,
  "risk_level": "High",
  "contributing_signals": [
    "Excessive spend deviation (30.0x personal 30-day baseline)",
    "High transaction velocity (3 transactions in trailing 1h)",
    "First-time access from unverified hardware device ID",
    "Cross-border transaction originating outside home jurisdiction",
    "High-risk merchant category (Wire Transfer)"
  ]
}


### 3. Set Up Three-Tier Action Decision Matrix
Routes payments through automated rules based on their risk score:

| Tier | Risk Score Range | Action | What Happens to the Customer |
| :--- | :---: | :--- | :--- |
| Low Risk | Score below 0.15 | Auto-Approve | Instant, seamless approval (~93% of all payments) |
| Medium Risk | 0.15 to Cutoff | Step-Up Verification | Asks customer for a quick one-time code (OTP) or biometric check |
| High Risk | At or above Cutoff | Analyst Review Queue | Payment is paused and sent to a fraud investigator |


In [5]:
def evaluate_triage(score, cutoff=0.79):
    if score < 0.15:
        return 'Auto-Approve (Low Risk)'
    elif score < cutoff:
        return 'Step-Up Verification (Medium Risk)'
    else:
        return 'Prioritized Analyst Review (High Risk)'

print(f'Score 0.05 -> {evaluate_triage(0.05)}')
print(f'Score 0.35 -> {evaluate_triage(0.35)}')
print(f'Score 0.88 -> {evaluate_triage(0.88)}')

Score 0.05 -> Auto-Approve (Low Risk)
Score 0.35 -> Step-Up Verification (Medium Risk)
Score 0.88 -> Prioritized Analyst Review (High Risk)


### 4. Review End-to-End System Architecture
Shows how data moves through the system from the moment a payment is requested to when an analyst reviews it:

```mermaid
graph TD
    TXN[Incoming Payment Request] --> FEAT[Feature Service: Spend Ratio, Velocity, Novelty]
    FEAT --> MODEL[Random Forest Inference Engine]
    MODEL --> SCORE[Continuous Fraud Probability (0.0 to 1.0)]
    SCORE --> TRIAGE{Risk Score Triage}
    TRIAGE -->|< 0.15| PASS[Auto-Approve: Instant Settlement]
    TRIAGE -->|0.15 to Cutoff| STEPUP[Step-Up Auth: OTP / Biometric Challenge]
    TRIAGE -->|>= Cutoff| QUEUE[Streamlit Prioritized Analyst Queue]
    QUEUE --> ANALYST[Analyst Decision: Confirm Fraud / False Positive]
    ANALYST --> STORE[(Audit Trail & Labeled Feedback Store)]
    STORE -.->|Monthly Retraining Loop| MODEL
```


### 5. Define Maintenance and Model Retraining Guidelines
Rules for keeping the fraud detection system healthy and accurate over time:

1. Daily Alert Check: If more than 4.0% of all payments get flagged in a single day, an alert sounds to check for coordinated fraud attacks or technical glitches.
2. Review Accuracy Check: Weekly reviews verify whether analysts agree with the model. If too many alerts are false alarms (confirmed fraud falls below 40%), the cutoff score is recalibrated.
3. Monthly Retraining: On the 1st of every month, the model is retrained using confirmed fraud and dispute records from the previous 30 days.
4. Emergency Retraining Trigger: If innocent payments get wrongly held more than 15.0% of the time two weeks in a row, the team rebuilds and updates the model immediately.


In [6]:
print('Governance Runbook Policies')
print('1. Queue Alert Tolerance: 1.5% - 4.0% of total payment volume')
print('2. Minimum Holdout ROC-AUC: >= 0.90')
print('3. Retraining Schedule: 1st calendar day of each month')
print('4. Emergency Trigger: FPR > 15% across 2 consecutive weeks')
print('Production governance specifications verified.')

Governance Runbook Policies
1. Queue Alert Tolerance: 1.5% - 4.0% of total payment volume
2. Minimum Holdout ROC-AUC: >= 0.90
3. Retraining Schedule: 1st calendar day of each month
4. Emergency Trigger: FPR > 15% across 2 consecutive weeks
Production governance specifications verified.


Runbook Summary:
- The live scoring function calculates instant risk ratings with clear, human-readable explanations.
- The 3-tier system stops fraud without slowing down innocent customers.
- Regular updates ensure the model stays smart as payment volumes grow and fraudsters try new tactics.
